In [ ]:
!sudo apt update -q
!sudo apt install -y pciutils zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

import threading, subprocess, time
thread = threading.Thread(target=lambda: subprocess.Popen(['ollama', 'serve']))
thread.start()
time.sleep(5)

!ollama pull qwen2.5:32b
!pip install ollama -q

!git clone https://github.com/average-peruvian/media-framing.git
!mv media-framing/* .
!rm -rf media-framing
!pip install -e . -q
!pip install hdbscan sentence-transformers -q

!wget -q https://github.com/average-peruvian/media-framing/releases/download/v0-alpha1/extracciones.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/ant/selection/out'

from medianalysis.factual.grift    import ExplodeEventsWorker
from medianalysis.factual.embed    import EventEmbedder
from medianalysis.factual.cluster  import cluster_generic
from medianalysis.factual.canonize import build_canon_input, EventCanonWorker

In [ ]:
# Explosión
ExplodeEventsWorker(
    input_csv='extracciones.csv',
    output_csv=f'{DRIVE_OUT}/eventos_raw.csv',
    id_col='id', batch_size=100, resume=True,
).run()

In [ ]:
# Embeddings
EventEmbedder(
    input_csv=f'{DRIVE_OUT}/eventos_raw.csv',
    output_csv=f'{DRIVE_OUT}/eventos_embeddings.csv',
    id_col='event_id', batch_size=500, resume=True,
).run()

In [ ]:
# Clustering
cluster_generic(
    embeddings_csv=f'{DRIVE_OUT}/eventos_embeddings.csv',
    id_col='event_id', output_csv=f'{DRIVE_OUT}/eventos_clusters.csv',
    prefix='evt', min_cluster_size=2,
)

In [ ]:
# Canonicalización
build_canon_input(
    clusters_csv=f'{DRIVE_OUT}/eventos_clusters.csv',
    raw_csv=f'{DRIVE_OUT}/eventos_raw.csv',
    id_col='event_id', text_cols=['event_type', 'trigger'],
    output_csv=f'{DRIVE_OUT}/eventos_para_canon.csv',
)

EventCanonWorker(
    input_csv=f'{DRIVE_OUT}/eventos_para_canon.csv',
    output_csv=f'{DRIVE_OUT}/tipos_evento.csv',
    id_col='cluster_id', batch_size=10, resume=True,
).run()